In [ ]:
import os
import pandas as pd
import requests

In [ ]:
# BASE_URL = "http://api.jolpi.ca/ergast/f1"

# def fetch_drivers(year):
#     url = f"{BASE_URL}/{year}/drivers"
#     response = requests.get(url)
#     if response.status_code == 200:
#         data = response.json()
#         drivers = data["MRData"]["DriverTable"]["Drivers"]
#         # Thêm cột "season" để biết dữ liệu thuộc năm nào
#         for driver in drivers:
#             driver["season"] = year
#         return drivers
#     else:
#         print(f"Failed to fetch data for {year}. Status: {response.status_code}")
#         return []

# # Danh sách chứa tất cả drivers từ 2010-2024
# all_drivers = []

# # Lấy dữ liệu từ 2010 đến 2024
# for year in range(2010, 2025):
#     print(f"Fetching drivers for year {year}...")
#     drivers = fetch_drivers(year)
#     all_drivers.extend(drivers)

# # Chuyển đổi dữ liệu sang DataFrame
# df_all_drivers = pd.DataFrame(all_drivers)

# # Lưu vào file CSV
# df_all_drivers.to_csv("drivers.csv", index=False)
# print("Saved all drivers to drivers.csv")

In [ ]:
BASE_URL = "http://api.jolpi.ca/ergast/f1"

In [ ]:
def get_season_data(api=BASE_URL + "/seasons", limit=30):
    all_data = []
    offset = 0
    total = 0
    
    while True:
        url = f"{api}?limit={limit}&offset={offset}"
        response = requests.get(url)

        if response.status_code == 200:
            data = (
                response.json()
                .get("MRData", {})
                .get("SeasonTable", {})
                .get("Seasons", [])
            )
            
            if not data:  
                break
            
            all_data.extend(data)
            offset += limit
            total = int(response.json()["MRData"]["total"])
            print(f"Fetching data {limit}/{offset}...")
            if offset >= total:
                break
            
            df = pd.DataFrame(data)
            os.makedirs("data2/season", exist_ok=True)
            df.to_csv("data2/season/seasons.csv", index=False)
        else:
            print(f"Failed to fetch data for {api}. Status: {response.status_code}")


get_season_data()

In [ ]:
{
    "season": "http://api.jolpi.ca/ergast/f1/seasons",
    "circuit": "http://api.jolpi.ca/ergast/f1/circuits",
    "status": "http://api.jolpi.ca/ergast/f1/status",
    
    "race": "http://api.jolpi.ca/ergast/f1/2024/races",
    "constructor": "http://api.jolpi.ca/ergast/f1/2024/constructors",
    "driver": "http://api.jolpi.ca/ergast/f1/2024/drivers",
    "result": "http://api.jolpi.ca/ergast/f1/2024/results",
    "sprint": "http://api.jolpi.ca/ergast/f1/2024/sprint",
    "qualifying": "http://api.jolpi.ca/ergast/f1/2024/qualifying",
    "driverstanding": "http://api.jolpi.ca/ergast/f1/2024/driverstandings",
    "constructorstanding": "http://api.jolpi.ca/ergast/f1/2024/constructorstandings",
    
    "pitstop": "http://api.jolpi.ca/ergast/f1/2024/1/pitstops",
    "lap": "http://api.jolpi.ca/ergast/f1/2024/1/laps",
}

In [ ]:
import requests
import pandas as pd
import os

def fetch_all_data(api_url, key_path, save_path, limit=30):
    """Crawl toàn bộ dữ liệu từ API có pagination và lưu vào CSV."""
    all_data = []
    offset = 0  

    while True:
        url = f"{api_url}?limit={limit}&offset={offset}"
        response = requests.get(url)

        if response.status_code != 200:
            print(f"❌ Lỗi khi fetch dữ liệu từ {url}. Status: {response.status_code}")
            break

        data = response.json()
        current_data = data

        # Truy cập vào đúng phần chứa danh sách dữ liệu
        for key in key_path:
            current_data = current_data.get(key, {})

        if not isinstance(current_data, list) or not current_data:
            break  # Dừng nếu không còn dữ liệu

        all_data.extend(current_data)
        offset += limit  # Tăng offset để lấy trang tiếp theo

    # Lưu dữ liệu vào file CSV
    if all_data:
        df = pd.DataFrame(all_data)
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        df.to_csv(save_path, index=False)
        print(f"✅ Lưu dữ liệu thành công: {save_path}")
    else:
        print("⚠️ Không có dữ liệu để lưu.")

# Crawl seasons
fetch_all_data(
    api_url="http://api.jolpi.ca/ergast/f1/seasons",
    key_path=["MRData", "SeasonTable", "Seasons"],
    save_path="data2/seasons.csv"
)

# Crawl circuits
fetch_all_data(
    api_url="http://api.jolpi.ca/ergast/f1/circuits",
    key_path=["MRData", "CircuitTable", "Circuits"],
    save_path="data2/circuits.csv"
)

# Crawl status
fetch_all_data(
    api_url="http://api.jolpi.ca/ergast/f1/status",
    key_path=["MRData", "StatusTable", "Status"],
    save_path="data2/status.csv"
)



✅ Lưu dữ liệu thành công: data2/seasons.csv
✅ Lưu dữ liệu thành công: data/circuits.csv
✅ Lưu dữ liệu thành công: data/status.csv


In [ ]:
def fetch_yearly_data(api_template, key_path, save_folder, start_year=1950, end_year=2024, limit=30):
    """Crawl dữ liệu theo từng năm (có pagination) và lưu vào file CSV riêng từng năm."""
    os.makedirs(save_folder, exist_ok=True)  # Tạo folder lưu dữ liệu nếu chưa có

    for year in range(start_year, end_year + 1):
        print(f"📅 Fetching data for year {year}...")

        all_data = []
        offset = 0  

        while True:
            url = f"{api_template.format(year=year)}?limit={limit}&offset={offset}"
            response = requests.get(url)

            if response.status_code != 200:
                print(f"❌ Lỗi khi fetch {url}. Status: {response.status_code}")
                break

            data = response.json()
            current_data = data

            # Truy cập vào phần chứa danh sách dữ liệu
            for key in key_path:
                current_data = current_data.get(key, {})

            if not isinstance(current_data, list) or not current_data:
                break  # Dừng nếu không còn dữ liệu

            all_data.extend(current_data)
            offset += limit  # Tăng offset để lấy trang tiếp theo

        # Lưu dữ liệu vào file CSV cho từng năm
        if all_data:
            df = pd.DataFrame(all_data)
            filename = os.path.join(save_folder, f"{year}.csv")
            df.to_csv(filename, index=False)
            print(f"✅ Lưu thành công: {filename}")
        else:
            print(f"⚠️ Không có dữ liệu cho năm {year}.")

# 📌 Ví dụ crawl dữ liệu `drivers`
fetch_yearly_data(
    api_template="http://api.jolpi.ca/ergast/f1/{year}/drivers",
    key_path=["MRData", "DriverTable", "Drivers"],
    save_dir="data/drivers"
)

# 📌 Tương tự cho `constructors`
fetch_yearly_data(
    api_template="http://api.jolpi.ca/ergast/f1/{year}/constructors",
    key_path=["MRData", "ConstructorTable", "Constructors"],
    save_dir="data/constructors"
)

In [ ]:
import requests
import pandas as pd
import os

BASE_URL = "http://api.jolpi.ca/ergast/f1"

def get_total_rounds(year):
    """Lấy số vòng đua (rounds) trong năm"""
    url = f"{BASE_URL}/{year}.json"
    response = requests.get(url)
    
    if response.status_code == 200:
        races = response.json().get("MRData", {}).get("RaceTable", {}).get("Races", [])
        return len(races)  # Trả về tổng số vòng đua trong năm
    else:
        print(f"❌ Không lấy được số vòng đua cho năm {year}")
        return 0

def fetch_race_data(api_template, key_path, save_dir, start_year=2000, end_year=2025):
    """Crawl dữ liệu theo mùa giải và từng vòng đua"""
    os.makedirs(save_dir, exist_ok=True)

    for year in range(start_year, end_year + 1):
        total_rounds = get_total_rounds(year)  # Kiểm tra có bao nhiêu vòng đua
        
        if total_rounds == 0:
            continue  # Bỏ qua năm nếu không có dữ liệu

        for round_num in range(1, total_rounds + 1):
            all_data = []
            offset = 0
            limit = 30  # Nếu có phân trang

            while True:
                url = f"{api_template.format(year=year, round=round_num)}?limit={limit}&offset={offset}"
                response = requests.get(url)

                if response.status_code != 200:
                    print(f"❌ Lỗi khi fetch {url}. Status: {response.status_code}")
                    break

                data = response.json()
                current_data = data

                # Truy cập vào phần chứa danh sách dữ liệu
                for key in key_path:
                    current_data = current_data.get(key, {})

                if not isinstance(current_data, list) or not current_data:
                    break  # Không còn dữ liệu -> Dừng

                all_data.extend(current_data)
                offset += limit  # Tăng offset để lấy trang tiếp theo

            # Lưu dữ liệu theo năm - vòng đua
            if all_data:
                save_path = os.path.join(save_dir, f"{year}_round{round_num}.csv")
                df = pd.DataFrame(all_data)
                df.to_csv(save_path, index=False)
                print(f"✅ Lưu {save_path}")
            else:
                print(f"⚠️ Không có dữ liệu cho {year} - Round {round_num}")

# 📌 Crawl `race results`
fetch_race_data(
    api_template=BASE_URL + "/{year}/{round}/results.json",
    key_path=["MRData", "RaceTable", "Races"],
    save_dir="data/race_results"
)

# 📌 Crawl `qualifying`
fetch_race_data(
    api_template=BASE_URL + "/{year}/{round}/qualifying.json",
    key_path=["MRData", "RaceTable", "Races"],
    save_dir="data/qualifying"
)

# 📌 Crawl `pitstops`
fetch_race_data(
    api_template=BASE_URL + "/{year}/{round}/pitstops.json",
    key_path=["MRData", "RaceTable", "Races"],
    save_dir="data/pitstops"
)


In [7]:
import requests
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter

# # API endpoint
# url = "https://api.jolpi.ca/ergast/"
# response = requests.get(url)

# if response.status_code == 200:
#     print("Data fetched successfully!")
#     data = response.json()  # Parse JSON response
# else:
#     print(f"Failed to fetch data. Status code: {response.status_code}")
#     exit()

# df = pd.DataFrame(data)
# csv_file = "ergast_data.csv"
# df.to_csv(csv_file, index=False)
# print(f"Data saved to {csv_file}")

file_path = "../KaggleData/"

# Load the latest version
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "rohanrao/formula-1-world-championship-1950-2020",
    file_path
)

print("First 5 records:", df.head())

C:\Users\quanghung\AppData\Local\Temp\ipykernel_23884\2875103692.py:25: DeprecationWarning: load_dataset is deprecated and will be removed in future version.
  df = kagglehub.load_dataset(


ValueError: Unsupported file extension: ''. Supported file extensions are: .csv, .tsv, .json, .jsonl, .xml, .parquet, .feather, .sqlite, .sqlite3, .db, .db3, .s3db, .dl3, .xls, .xlsx, .xlsm, .xlsb, .odf, .ods, .odt